In [ ]:
import torch

from torch import nn
import torchvision

class seq(nn.Module):
    def __init__(self):
        super(seq, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, padding=2, stride=1)
        # 假设输入图像尺寸为32*32，输入通道数为3（RGB图像），则按以上设置卷积核大小为5*5，输出通道数为32，步长为1，填充为2，
        # 得到输出尺寸不变，仍为32*32，
        # 计算公式：output_size = (input_size - kernel_size + 2 * padding) / stride + 1
        # 注意区分in_channels和input_size，in_channels是输入的通道数，input_size是输入的空间尺寸（宽和高）
        # 32 = (32 - 5 + 2 * padding) / 1 + 1
        # padding=2，kernel_size=5，stride=1满足输出尺寸不变的条件：padding = (kernel_size - 1) / 2

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, padding=2, stride=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, padding=2, stride=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        # flatten将输入的64张4*4的特征图展平为一维向量，长度为64*4*4=1024
        # 1024 = 64 * 4 * 4
        self.linear1 = nn.Linear(1024, 64)
        # 所谓线性层，就是全连接层，输入的每个元素都与输出的每个元素相连，输出的每个元素都是输入的所有元素的线性组合
        self.linear2 = nn.Linear(64, 10)


        self.model1 = nn.Sequential(
            self.conv1,
            self.pool1,
            self.conv2,
            self.pool2,
            self.conv3,
            self.pool3,
            self.flatten,
            self.linear1,
            self.linear2
        )


    def forward(self, input):
        # output = self.conv1(input)
        # output = self.pool1(output)
        # output = self.conv2(output)
        # output = self.pool2(output)
        # output = self.conv3(output)
        # output = self.pool3(output)
        # output = self.flatten(output)
        # output = self.linear1(output)
        # output = self.linear2(output)
        output = self.model1(input)

        return output


dataset = torchvision.datasets.CIFAR10(root="./dataset", train=True, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

net = seq()

optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

loss = nn.CrossEntropyLoss()
for data in dataloader:
    inputs, targets = data
    outputs = net(inputs)
    result_loss = loss(outputs, targets)
    # print(outputs)
    # print(targets)
    # print(result_loss)
    optimizer.zero_grad()   # 在反向传播之前，先将梯度清零，否则会累积上一次的梯度
    
    #print(net.parameters())  # 打印网络的参数，应该是一个可迭代的对象，包含了网络中所有可学习的参数，例如卷积层的权重和偏置，线性层的权重和偏置等
    
    result_loss.backward()  # backward()函数会自动计算每个参数的梯度，并将其存储在参数的.grad属性中
    
    #print(net.parameters)  # 打印每个参数的梯度，经过反向传播后应该有具体的数值，表示每个参数的梯度

    optimizer.step()  # step()函数会根据参数的梯度更新参数的值，更新的方式取决于优化器的算法，例如SGD会按照学习率和梯度的乘积来更新参数 

    #optimizer.zero_grad()  这行没有必要，首尾二选一，放在反向传播之前或之后都可以，通常放在反向传播之前更常见，因为这样可以确保每次迭代开始时梯度都是清零的状态，避免了上一次迭代的梯度对当前迭代的影响






Files already downloaded and verified
